
# 15 — Country POSet Diagnostic Drilldown  
## Explaining a Country's Position in the Final 5-Variable POSet

This notebook creates country-level diagnostic drilldowns for selected countries.

It runs after:

```text
14_Fattore_Style_POSet_Scores_2002_5Var.ipynb
```

## Why this notebook exists

The final POSet gives a structural map, but the report may need country-level explanations such as:

- Why is a country in a frontier or non-frontier layer?
- Which profiles/countries dominate it?
- Which profiles/countries does it dominate?
- Which variables explain its position?
- Is it incomparable with many countries?
- Does its structural position align with recovery outcomes?

## Default target countries

The default target list is:

```text
DEU, ITA, FRA, CAN, USA
```

You can edit `TARGET_COUNTRIES` in the configuration cell.

## Important interpretation guardrail

This drilldown explains POSet position.  
It does **not** create a final scalar ranking or Economic Resilience Index.

## Main outputs

- `country_poset_diagnostic_drilldown.csv`
- `target_country_problem_brief.csv`
- `target_direct_hasse_parents.csv`
- `target_direct_hasse_children.csv`
- `target_dominators_summary.csv`
- `target_dominated_summary.csv`
- `target_incomparability_summary.csv`
- `target_variable_diagnostics.csv`
- `target_dominators_variable_explanation.csv`
- `target_recovery_alignment.csv`
- `report_ready_country_drilldown_paragraphs.csv`
- `15_Country_POSet_Diagnostic_Drilldown_Audit.xlsx`


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

MASTER_DIR = PROJECT_ROOT / "Data" / "Processed" / "Master_Dataset"
PROFILE_DIR = PROJECT_ROOT / "Data" / "Processed" / "Profile_POSet_Main"
FATTORE_DIR = PROJECT_ROOT / "Data" / "Processed" / "Fattore_Style_POSet_Scores"
RECOVERY_DIR = PROJECT_ROOT / "Data" / "Processed" / "Recovery_Validation"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Country_POSet_Diagnostic_Drilldown"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "15_Country_POSet_Diagnostic_Drilldown"
FIGURES_DIR = PROJECT_ROOT / "Outputs" / "Figures" / "Country_POSet_Diagnostic_Drilldown"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Country_POSet_Diagnostic_Drilldown"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, FIGURES_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())


Run ID: 20260624_215532
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Country_POSet_Diagnostic_Drilldown


In [2]:

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

TARGET_COUNTRIES = ["DEU", "ITA", "FRA", "CAN", "USA"]

SHOCK_IDS = ["2007", "2019"]

MAIN_VARIABLE_SET = "baseline_5_variables"
MAIN_LEVELS = 5

LEVEL_COLUMNS = [
    "debt_capacity_level_5",
    "employment_strength_level_5",
    "rd_intensity_level_5",
    "human_capital_tertiary_level_5",
    "energy_security_level_5",
]

SCORE_COLUMNS = [
    "debt_capacity_score_0_100",
    "employment_strength_score_0_100",
    "rd_intensity_score_0_100",
    "human_capital_tertiary_score_0_100",
    "energy_security_score_0_100",
]

VARIABLE_LABELS = {
    "debt_capacity_score_0_100": "Debt capacity",
    "employment_strength_score_0_100": "Employment strength",
    "rd_intensity_score_0_100": "R&D intensity",
    "human_capital_tertiary_score_0_100": "Human capital",
    "energy_security_score_0_100": "Energy security",
    "debt_capacity_level_5": "Debt capacity",
    "employment_strength_level_5": "Employment strength",
    "rd_intensity_level_5": "R&D intensity",
    "human_capital_tertiary_level_5": "Human capital",
    "energy_security_level_5": "Energy security",
}

INPUT_FILES = {
    "profile_country_map": PROFILE_DIR / "profile_country_map_with_layers.csv",
    "profile_nodes": PROFILE_DIR / "profile_nodes_with_layers.csv",
    "dominance_relations": PROFILE_DIR / "profile_dominance_relations.csv",
    "hasse_edges": PROFILE_DIR / "profile_hasse_edges.csv",
    "fattore_country_scores": FATTORE_DIR / "fattore_country_scores.csv",
    "structural_complete_cases": MASTER_DIR / "structural_snapshot_final5_complete_cases.csv",
    "recovery_validation_country_dataset": RECOVERY_DIR / "recovery_validation_country_dataset.csv",
}

required_inputs = [
    "profile_country_map",
    "profile_nodes",
    "dominance_relations",
    "hasse_edges",
    "fattore_country_scores",
    "structural_complete_cases",
]

missing_inputs = [name for name in required_inputs if not INPUT_FILES[name].exists()]

if missing_inputs:
    raise FileNotFoundError(
        "Missing required country drilldown input files:\n"
        + "\n".join([f"- {name}: {INPUT_FILES[name]}" for name in missing_inputs])
    )

print("Target countries:", TARGET_COUNTRIES)
print("Shock IDs:", SHOCK_IDS)


Target countries: ['DEU', 'ITA', 'FRA', 'CAN', 'USA']
Shock IDs: ['2007', '2019']


In [3]:

# ------------------------------------------------------
# HELPERS
# ------------------------------------------------------

table_inventory = []
figure_inventory = []

def clean_keys(df):
    out = df.copy()

    if "Country" in out.columns:
        out["Country"] = out["Country"].astype(str).str.strip().str.upper()

    if "shock_id" in out.columns:
        out["shock_id"] = out["shock_id"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()

    if "baseline_year" in out.columns:
        out["baseline_year"] = pd.to_numeric(out["baseline_year"], errors="coerce").astype("Int64")

    return out


def save_table(df, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    df.to_csv(processed_path, index=False)
    df.to_csv(diagnostics_path, index=False)
    df.to_csv(table_path, index=False)

    table_inventory.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(df),
        "columns": len(df.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved table:", file_name)


def save_figure(fig, file_name, title, description):
    path = FIGURES_DIR / file_name
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)

    figure_inventory.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "path": str(path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved figure:", file_name)


def dominates(levels_a, levels_b):
    a = np.asarray(levels_a, dtype=int)
    b = np.asarray(levels_b, dtype=int)
    return bool(np.all(a >= b) and np.any(a > b))


def relation_between_profiles(profile_a_levels, profile_b_levels):
    if dominates(profile_a_levels, profile_b_levels):
        return "A_dominates_B"
    if dominates(profile_b_levels, profile_a_levels):
        return "B_dominates_A"
    return "incomparable"


def summarize_dimension_gaps(target_row, other_row):
    gaps = []

    for col in LEVEL_COLUMNS:
        label = VARIABLE_LABELS.get(col, col)
        target_value = int(target_row[col])
        other_value = int(other_row[col])
        diff = target_value - other_value

        if diff > 0:
            direction = "target_stronger"
        elif diff < 0:
            direction = "target_weaker"
        else:
            direction = "equal"

        gaps.append({
            "dimension": label,
            "target_level": target_value,
            "other_level": other_value,
            "target_minus_other": diff,
            "dimension_relation": direction,
        })

    return gaps


In [4]:

# ------------------------------------------------------
# LOAD INPUT TABLES
# ------------------------------------------------------

profile_country_map = clean_keys(pd.read_csv(INPUT_FILES["profile_country_map"]))
profile_nodes = clean_keys(pd.read_csv(INPUT_FILES["profile_nodes"]))
dominance_relations = clean_keys(pd.read_csv(INPUT_FILES["dominance_relations"]))
hasse_edges = clean_keys(pd.read_csv(INPUT_FILES["hasse_edges"]))
fattore_country_scores = clean_keys(pd.read_csv(INPUT_FILES["fattore_country_scores"]))
structural = clean_keys(pd.read_csv(INPUT_FILES["structural_complete_cases"]))

if INPUT_FILES["recovery_validation_country_dataset"].exists():
    recovery_country = clean_keys(pd.read_csv(INPUT_FILES["recovery_validation_country_dataset"]))
else:
    recovery_country = pd.DataFrame()

for df in [profile_country_map, profile_nodes, fattore_country_scores, structural, recovery_country]:
    for col in LEVEL_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    for col in SCORE_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

print("profile_country_map:", profile_country_map.shape)
print("profile_nodes:", profile_nodes.shape)
print("dominance_relations:", dominance_relations.shape)
print("hasse_edges:", hasse_edges.shape)
print("fattore_country_scores:", fattore_country_scores.shape)
print("structural:", structural.shape)
print("recovery_country:", recovery_country.shape)

display(profile_country_map.head())


profile_country_map: (60, 19)
profile_nodes: (59, 21)
dominance_relations: (225, 9)
hasse_edges: (116, 10)
fattore_country_scores: (60, 28)
structural: (60, 46)
recovery_country: (60, 26)


,Country,shock_id,baseline_year,profile_code,profile_digit_sum,debt_capacity_level_5,employment_strength_level_5,rd_intensity_level_5,human_capital_tertiary_level_5,energy_security_level_5,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100,main_variable_set,levels,layer,is_pareto_frontier
0,AUT,2007,2007,2-4-5-3-2,16.0000,2,4,5,3,2,38.5303,80.6046,68.7303,33.2757,22.6201,baseline_5_variables,5,2,False
1,BEL,2007,2007,1-2-4-4-1,12.0000,1,2,4,4,1,16.9811,46.5777,48.5360,53.5113,9.6139,baseline_5_variables,5,3,False
2,CAN,2007,2007,3-3-4-5-5,20.0000,3,3,4,5,5,64.9865,63.6697,50.3901,100.0000,100.0000,baseline_5_variables,5,1,True
3,CZE,2007,2007,4-3-3-1-5,16.0000,4,3,3,1,5,76.7627,74.6107,29.3904,0.4466,50.5562,baseline_5_variables,5,2,False
4,DEU,2007,2007,2-1-4-2-3,12.0000,2,1,4,2,3,40.6157,31.1478,68.2315,30.9850,28.5833,baseline_5_variables,5,4,False


In [5]:

# ------------------------------------------------------
# BUILD ENRICHED COUNTRY DRILLDOWN BASE
# ------------------------------------------------------

base = profile_country_map.copy()

# Merge Fattore scores.
score_cols = [
    "Country", "shock_id", "baseline_year", "profile_code",
    "observed_dominance_score_normalized",
    "observed_dominated_by_score_normalized",
    "observed_incomparability_score_normalized",
    "embedded_dominance_score_normalized",
    "embedded_dominated_by_score_normalized",
    "embedded_incomparability_score_normalized",
    "observed_separation_score_normalized",
    "embedded_separation_score_normalized",
]

score_cols = [c for c in score_cols if c in fattore_country_scores.columns]

base = base.merge(
    fattore_country_scores[score_cols],
    on=["Country", "shock_id", "baseline_year", "profile_code"],
    how="left",
)

# Merge recovery outcome.
if len(recovery_country):
    recovery_cols = [
        "Country", "shock_id", "baseline_year",
        "recovery_years", "recovery_status",
    ]
    recovery_cols = [c for c in recovery_cols if c in recovery_country.columns]

    base = base.merge(
        recovery_country[recovery_cols],
        on=["Country", "shock_id", "baseline_year"],
        how="left",
    )
else:
    base["recovery_years"] = np.nan
    base["recovery_status"] = np.nan

base["target_flag"] = base["Country"].isin(TARGET_COUNTRIES)
base["layer"] = pd.to_numeric(base["layer"], errors="coerce").astype("Int64")

save_table(
    base,
    "country_poset_diagnostic_drilldown.csv",
    "Country POSet diagnostic drilldown",
    "Country-level POSet position, profile, diagnostic scores, and recovery outcomes.",
)

target_base = base[base["target_flag"]].copy()

save_table(
    target_base,
    "target_country_problem_brief.csv",
    "Target country problem brief",
    "Compact profile/layer/frontier/recovery summary for configured target countries.",
)

display(target_base.head(20))


Saved table: country_poset_diagnostic_drilldown.csv
Saved table: target_country_problem_brief.csv


,Country,shock_id,baseline_year,profile_code,profile_digit_sum,debt_capacity_level_5,employment_strength_level_5,rd_intensity_level_5,human_capital_tertiary_level_5,energy_security_level_5,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100,main_variable_set,levels,layer,is_pareto_frontier,observed_dominance_score_normalized,observed_dominated_by_score_normalized,observed_incomparability_score_normalized,embedded_dominance_score_normalized,embedded_dominated_by_score_normalized,embedded_incomparability_score_normalized,observed_separation_score_normalized,embedded_separation_score_normalized,recovery_years,recovery_status,target_flag
2,CAN,2007,2007,3-3-4-5-5,20.0000,3,3,4,5,5,64.9865,63.6697,50.3901,100.0000,100.0000,baseline_5_variables,5,1,True,0.3333,0.0000,0.6667,0.2878,0.0054,0.7068,0.3060,0.1607,1.0000,recovered,True
4,DEU,2007,2007,2-1-4-2-3,12.0000,2,1,4,2,3,40.6157,31.1478,68.2315,30.9850,28.5833,baseline_5_variables,5,4,False,0.0417,0.2917,0.6667,0.0150,0.1533,0.8316,0.0030,0.0016,2.0000,recovered,True
9,FRA,2007,2007,2-2-4-3-3,14.0000,2,2,4,3,3,38.8282,39.5629,54.7007,37.7650,33.9095,baseline_5_variables,5,3,False,0.1667,0.2500,0.5833,0.0458,0.0919,0.8624,0.0243,0.0128,2.0000,recovered,True
14,ITA,2007,2007,1-3-2-1-2,9.0000,1,3,2,1,2,1.0924,64.6774,23.4216,0.0000,13.0036,baseline_5_variables,5,3,False,0.0417,0.4167,0.5417,0.0035,0.3838,0.6127,0.0007,0.0004,15.0000,recovered,True
24,USA,2007,2007,2-4-5-5-4,20.0000,2,4,5,5,4,48.4466,83.6932,75.0128,77.1390,47.1942,baseline_5_variables,5,1,True,0.3333,0.0000,0.6667,0.2558,0.0048,0.7394,0.2828,0.1485,1.0000,recovered,True
29,CAN,2019,2019,3-3-3-5-5,19.0000,3,3,3,5,5,74.5337,86.1115,35.4737,100.0000,54.4865,baseline_5_variables,5,1,True,0.2424,0.0000,0.7576,0.2157,0.0083,0.7759,0.4054,0.1009,1.0000,recovered,True
33,DEU,2019,2019,3-5-5-2-2,17.0000,3,5,5,2,2,71.4696,95.7551,68.9906,32.6268,11.6425,baseline_5_variables,5,1,True,0.1212,0.0000,0.8788,0.0957,0.0150,0.8892,0.1264,0.0315,2.0000,recovered,True
38,FRA,2019,2019,1-1-4-3-3,12.0000,1,1,4,3,3,48.7945,75.6772,46.3824,51.1465,17.5878,baseline_5_variables,5,3,False,0.0606,0.2121,0.7273,0.0112,0.1437,0.8451,0.0120,0.0030,2.0000,recovered,True
44,ITA,2019,2019,1-1-3-1-1,7.0000,1,1,3,1,1,28.3008,70.2293,28.0464,9.0169,8.0242,baseline_5_variables,5,4,False,0.0000,0.5758,0.4242,0.0006,0.5999,0.3995,0.0000,0.0000,2.0000,recovered,True
58,USA,2019,2019,1-5-5-5-5,21.0000,1,5,5,5,5,47.1987,93.7424,69.7953,75.2903,32.4693,baseline_5_variables,5,1,True,0.1818,0.0000,0.8182,0.1997,0.0013,0.7990,0.3809,0.0948,1.0000,recovered,True


In [6]:

# ------------------------------------------------------
# DIRECT HASSE PARENTS AND CHILDREN FOR TARGET COUNTRIES
# ------------------------------------------------------

target_profiles = target_base[
    ["Country", "shock_id", "baseline_year", "profile_code"]
].drop_duplicates()

parents_rows = []
children_rows = []

for _, target in target_profiles.iterrows():
    shock_id = str(target["shock_id"])
    profile_code = target["profile_code"]

    parent_edges = hasse_edges[
        hasse_edges["shock_id"].astype(str).eq(shock_id)
        & hasse_edges["dominated_profile"].astype(str).eq(str(profile_code))
    ].copy()

    child_edges = hasse_edges[
        hasse_edges["shock_id"].astype(str).eq(shock_id)
        & hasse_edges["dominating_profile"].astype(str).eq(str(profile_code))
    ].copy()

    for _, edge in parent_edges.iterrows():
        parent_profile = edge["dominating_profile"]
        parent_countries = profile_nodes[
            profile_nodes["shock_id"].astype(str).eq(shock_id)
            & profile_nodes["profile_code"].astype(str).eq(str(parent_profile))
        ]["countries"]

        parents_rows.append({
            "target_country": target["Country"],
            "shock_id": shock_id,
            "baseline_year": target["baseline_year"],
            "target_profile": profile_code,
            "parent_profile_directly_dominates_target": parent_profile,
            "parent_countries": parent_countries.iloc[0] if len(parent_countries) else "",
            "edge_type": edge.get("edge_type", "hasse_cover_relation"),
        })

    for _, edge in child_edges.iterrows():
        child_profile = edge["dominated_profile"]
        child_countries = profile_nodes[
            profile_nodes["shock_id"].astype(str).eq(shock_id)
            & profile_nodes["profile_code"].astype(str).eq(str(child_profile))
        ]["countries"]

        children_rows.append({
            "target_country": target["Country"],
            "shock_id": shock_id,
            "baseline_year": target["baseline_year"],
            "target_profile": profile_code,
            "child_profile_directly_dominated_by_target": child_profile,
            "child_countries": child_countries.iloc[0] if len(child_countries) else "",
            "edge_type": edge.get("edge_type", "hasse_cover_relation"),
        })

target_direct_hasse_parents = pd.DataFrame(parents_rows)
target_direct_hasse_children = pd.DataFrame(children_rows)

if target_direct_hasse_parents.empty:
    target_direct_hasse_parents = pd.DataFrame(columns=[
        "target_country", "shock_id", "baseline_year", "target_profile",
        "parent_profile_directly_dominates_target", "parent_countries", "edge_type"
    ])

if target_direct_hasse_children.empty:
    target_direct_hasse_children = pd.DataFrame(columns=[
        "target_country", "shock_id", "baseline_year", "target_profile",
        "child_profile_directly_dominated_by_target", "child_countries", "edge_type"
    ])

save_table(
    target_direct_hasse_parents,
    "target_direct_hasse_parents.csv",
    "Target direct Hasse parents",
    "Direct Hasse parent profiles that dominate target country profiles.",
)

save_table(
    target_direct_hasse_children,
    "target_direct_hasse_children.csv",
    "Target direct Hasse children",
    "Direct Hasse child profiles dominated by target country profiles.",
)

display(target_direct_hasse_parents.head(30))
display(target_direct_hasse_children.head(30))


Saved table: target_direct_hasse_parents.csv
Saved table: target_direct_hasse_children.csv


,target_country,shock_id,baseline_year,target_profile,parent_profile_directly_dominates_target,parent_countries,edge_type
0,DEU,2007,2007,2-1-4-2-3,2-2-4-3-3,FRA,hasse_cover_relation
1,FRA,2007,2007,2-2-4-3-3,2-4-5-5-4,USA,hasse_cover_relation
2,FRA,2007,2007,2-2-4-3-3,3-3-4-5-5,CAN,hasse_cover_relation
3,FRA,2007,2007,2-2-4-3-3,3-5-4-4-4,NLD,hasse_cover_relation
4,FRA,2007,2007,2-2-4-3-3,3-3-5-4-4,SWE,hasse_cover_relation
5,FRA,2007,2007,2-2-4-3-3,3-2-5-5-3,FIN,hasse_cover_relation
6,ITA,2007,2007,1-3-2-1-2,5-5-2-5-5,EST,hasse_cover_relation
7,ITA,2007,2007,1-3-2-1-2,3-3-4-5-5,CAN,hasse_cover_relation
8,ITA,2007,2007,1-3-2-1-2,3-5-4-4-4,NLD,hasse_cover_relation
9,ITA,2007,2007,1-3-2-1-2,3-3-5-4-4,SWE,hasse_cover_relation


,target_country,shock_id,baseline_year,target_profile,child_profile_directly_dominated_by_target,child_countries,edge_type
0,CAN,2007,2007,3-3-4-5-5,2-2-4-3-3,FRA,hasse_cover_relation
1,CAN,2007,2007,3-3-4-5-5,1-2-4-4-1,BEL,hasse_cover_relation
2,CAN,2007,2007,3-3-4-5-5,3-1-1-2-4,POL,hasse_cover_relation
3,CAN,2007,2007,3-3-4-5-5,1-3-2-1-2,ITA,hasse_cover_relation
4,DEU,2007,2007,2-1-4-2-3,1-1-1-2-2,GRC,hasse_cover_relation
5,FRA,2007,2007,2-2-4-3-3,2-1-4-2-3,DEU,hasse_cover_relation
6,FRA,2007,2007,2-2-4-3-3,2-2-2-1-3,HUN,hasse_cover_relation
7,ITA,2007,2007,1-3-2-1-2,1-2-2-1-1,PRT,hasse_cover_relation
8,USA,2007,2007,2-4-5-5-4,2-4-5-3-2,AUT,hasse_cover_relation
9,USA,2007,2007,2-4-5-5-4,2-2-4-3-3,FRA,hasse_cover_relation


In [7]:

# ------------------------------------------------------
# FULL DOMINATORS, DOMINATED, AND INCOMPARABLE PROFILES
# ------------------------------------------------------

dominator_rows = []
dominated_rows = []
incomparability_rows = []
variable_gap_rows = []

for _, target in target_profiles.iterrows():
    shock_id = str(target["shock_id"])
    target_country = target["Country"]
    target_profile = str(target["profile_code"])

    target_node = profile_nodes[
        profile_nodes["shock_id"].astype(str).eq(shock_id)
        & profile_nodes["profile_code"].astype(str).eq(target_profile)
    ]

    if target_node.empty:
        continue

    target_node = target_node.iloc[0]

    shock_nodes = profile_nodes[profile_nodes["shock_id"].astype(str).eq(shock_id)].copy()

    target_levels = [int(target_node[c]) for c in LEVEL_COLUMNS]

    for _, other in shock_nodes.iterrows():
        other_profile = str(other["profile_code"])

        if other_profile == target_profile:
            continue

        other_levels = [int(other[c]) for c in LEVEL_COLUMNS]
        relation = relation_between_profiles(other_levels, target_levels)

        if relation == "A_dominates_B":
            dominator_rows.append({
                "target_country": target_country,
                "shock_id": shock_id,
                "target_profile": target_profile,
                "dominator_profile": other_profile,
                "dominator_countries": other.get("countries", ""),
                "dominator_layer": other.get("layer", np.nan),
            })

            for gap in summarize_dimension_gaps(target_node, other):
                variable_gap_rows.append({
                    "target_country": target_country,
                    "shock_id": shock_id,
                    "comparison_type": "target_vs_dominator",
                    "target_profile": target_profile,
                    "other_profile": other_profile,
                    "other_countries": other.get("countries", ""),
                    **gap,
                })

        elif relation == "B_dominates_A":
            dominated_rows.append({
                "target_country": target_country,
                "shock_id": shock_id,
                "target_profile": target_profile,
                "dominated_profile": other_profile,
                "dominated_countries": other.get("countries", ""),
                "dominated_layer": other.get("layer", np.nan),
            })

            for gap in summarize_dimension_gaps(target_node, other):
                variable_gap_rows.append({
                    "target_country": target_country,
                    "shock_id": shock_id,
                    "comparison_type": "target_vs_dominated",
                    "target_profile": target_profile,
                    "other_profile": other_profile,
                    "other_countries": other.get("countries", ""),
                    **gap,
                })

        else:
            incomparability_rows.append({
                "target_country": target_country,
                "shock_id": shock_id,
                "target_profile": target_profile,
                "incomparable_profile": other_profile,
                "incomparable_countries": other.get("countries", ""),
                "incomparable_layer": other.get("layer", np.nan),
            })

            for gap in summarize_dimension_gaps(target_node, other):
                variable_gap_rows.append({
                    "target_country": target_country,
                    "shock_id": shock_id,
                    "comparison_type": "target_vs_incomparable",
                    "target_profile": target_profile,
                    "other_profile": other_profile,
                    "other_countries": other.get("countries", ""),
                    **gap,
                })

target_dominators_summary = pd.DataFrame(dominator_rows)
target_dominated_summary = pd.DataFrame(dominated_rows)
target_incomparability_summary = pd.DataFrame(incomparability_rows)
target_pairwise_variable_gaps = pd.DataFrame(variable_gap_rows)

for df_name, df_obj, cols in [
    ("target_dominators_summary", target_dominators_summary, ["target_country", "shock_id", "target_profile", "dominator_profile", "dominator_countries", "dominator_layer"]),
    ("target_dominated_summary", target_dominated_summary, ["target_country", "shock_id", "target_profile", "dominated_profile", "dominated_countries", "dominated_layer"]),
    ("target_incomparability_summary", target_incomparability_summary, ["target_country", "shock_id", "target_profile", "incomparable_profile", "incomparable_countries", "incomparable_layer"]),
    ("target_pairwise_variable_gaps", target_pairwise_variable_gaps, ["target_country", "shock_id", "comparison_type", "target_profile", "other_profile", "other_countries", "dimension", "target_level", "other_level", "target_minus_other", "dimension_relation"]),
]:
    if df_obj.empty:
        locals()[df_name] = pd.DataFrame(columns=cols)

save_table(
    target_dominators_summary,
    "target_dominators_summary.csv",
    "Target dominators summary",
    "All profiles that dominate each target country profile.",
)

save_table(
    target_dominated_summary,
    "target_dominated_summary.csv",
    "Target dominated summary",
    "All profiles dominated by each target country profile.",
)

save_table(
    target_incomparability_summary,
    "target_incomparability_summary.csv",
    "Target incomparability summary",
    "All profiles incomparable with each target country profile.",
)

save_table(
    target_pairwise_variable_gaps,
    "target_pairwise_variable_gaps.csv",
    "Target pairwise variable gaps",
    "Dimension-level ordinal gaps between target profiles and related profiles.",
)

display(target_dominators_summary.head(20))
display(target_incomparability_summary.head(20))


Saved table: target_dominators_summary.csv
Saved table: target_dominated_summary.csv
Saved table: target_incomparability_summary.csv
Saved table: target_pairwise_variable_gaps.csv


,target_country,shock_id,target_profile,dominator_profile,dominator_countries,dominator_layer
0,DEU,2007,2-1-4-2-3,4-5-5-4-5,DNK,1
1,DEU,2007,2-1-4-2-3,2-4-5-5-4,USA,1
2,DEU,2007,2-1-4-2-3,3-3-4-5-5,CAN,1
3,DEU,2007,2-1-4-2-3,3-5-4-4-4,NLD,2
4,DEU,2007,2-1-4-2-3,3-3-5-4-4,SWE,2
5,DEU,2007,2-1-4-2-3,3-2-5-5-3,FIN,1
6,DEU,2007,2-1-4-2-3,2-2-4-3-3,FRA,3
7,FRA,2007,2-2-4-3-3,4-5-5-4-5,DNK,1
8,FRA,2007,2-2-4-3-3,2-4-5-5-4,USA,1
9,FRA,2007,2-2-4-3-3,3-3-4-5-5,CAN,1


,target_country,shock_id,target_profile,incomparable_profile,incomparable_countries,incomparable_layer
0,CAN,2007,3-3-4-5-5,4-5-5-4-5,DNK,1
1,CAN,2007,3-3-4-5-5,5-5-2-5-5,EST,1
2,CAN,2007,3-3-4-5-5,2-4-5-5-4,USA,1
3,CAN,2007,3-3-4-5-5,3-5-4-4-4,NLD,2
4,CAN,2007,3-3-4-5-5,3-3-5-4-4,SWE,2
5,CAN,2007,3-3-4-5-5,1-4-3-5-5,GBR,1
6,CAN,2007,3-3-4-5-5,3-2-5-5-3,FIN,1
7,CAN,2007,3-3-4-5-5,5-4-3-2-4,SVN,1
8,CAN,2007,3-3-4-5-5,5-5-1-3-3,LTU,2
9,CAN,2007,3-3-4-5-5,5-5-3-3-1,LUX,1


In [8]:

# ------------------------------------------------------
# VARIABLE DIAGNOSTICS FOR TARGET COUNTRIES
# ------------------------------------------------------

variable_rows = []

for _, row in target_base.iterrows():
    for score_col, level_col in zip(SCORE_COLUMNS, LEVEL_COLUMNS):
        variable_rows.append({
            "target_country": row["Country"],
            "shock_id": row["shock_id"],
            "baseline_year": row["baseline_year"],
            "profile_code": row["profile_code"],
            "layer": row["layer"],
            "dimension": VARIABLE_LABELS.get(score_col, score_col),
            "score_0_100": row.get(score_col, np.nan),
            "ordinal_level_1_to_5": row.get(level_col, np.nan),
            "interpretation": (
                "Level 5 is strongest and level 1 is weakest within the shock-specific baseline sample."
            ),
        })

target_variable_diagnostics = pd.DataFrame(variable_rows)

# Summarize weaknesses against dominators.
dominator_gaps = target_pairwise_variable_gaps[
    target_pairwise_variable_gaps["comparison_type"].eq("target_vs_dominator")
].copy()

if len(dominator_gaps):
    target_dominators_variable_explanation = (
        dominator_gaps
        .groupby(["target_country", "shock_id", "dimension", "dimension_relation"])
        .agg(
            comparisons=("other_profile", "nunique"),
            mean_target_minus_other=("target_minus_other", "mean"),
            min_target_minus_other=("target_minus_other", "min"),
            max_target_minus_other=("target_minus_other", "max"),
        )
        .reset_index()
        .sort_values(["target_country", "shock_id", "mean_target_minus_other"])
    )
else:
    target_dominators_variable_explanation = pd.DataFrame(columns=[
        "target_country", "shock_id", "dimension", "dimension_relation",
        "comparisons", "mean_target_minus_other", "min_target_minus_other",
        "max_target_minus_other",
    ])

save_table(
    target_variable_diagnostics,
    "target_variable_diagnostics.csv",
    "Target variable diagnostics",
    "Final five score/ordinal-level diagnostics for each target country.",
)

save_table(
    target_dominators_variable_explanation,
    "target_dominators_variable_explanation.csv",
    "Target dominators variable explanation",
    "Dimension-level explanation of where target countries are weaker/equal/stronger than dominating profiles.",
)

display(target_variable_diagnostics.head(30))
display(target_dominators_variable_explanation.head(30))


Saved table: target_variable_diagnostics.csv
Saved table: target_dominators_variable_explanation.csv


,target_country,shock_id,baseline_year,profile_code,layer,dimension,score_0_100,ordinal_level_1_to_5,interpretation
0,CAN,2007,2007,3-3-4-5-5,1,Debt capacity,64.9865,3,Level 5 is strongest and level 1 is weakest wi...
1,CAN,2007,2007,3-3-4-5-5,1,Employment strength,63.6697,3,Level 5 is strongest and level 1 is weakest wi...
2,CAN,2007,2007,3-3-4-5-5,1,R&D intensity,50.3901,4,Level 5 is strongest and level 1 is weakest wi...
3,CAN,2007,2007,3-3-4-5-5,1,Human capital,100.0000,5,Level 5 is strongest and level 1 is weakest wi...
4,CAN,2007,2007,3-3-4-5-5,1,Energy security,100.0000,5,Level 5 is strongest and level 1 is weakest wi...
5,DEU,2007,2007,2-1-4-2-3,4,Debt capacity,40.6157,2,Level 5 is strongest and level 1 is weakest wi...
6,DEU,2007,2007,2-1-4-2-3,4,Employment strength,31.1478,1,Level 5 is strongest and level 1 is weakest wi...
7,DEU,2007,2007,2-1-4-2-3,4,R&D intensity,68.2315,4,Level 5 is strongest and level 1 is weakest wi...
8,DEU,2007,2007,2-1-4-2-3,4,Human capital,30.9850,2,Level 5 is strongest and level 1 is weakest wi...
9,DEU,2007,2007,2-1-4-2-3,4,Energy security,28.5833,3,Level 5 is strongest and level 1 is weakest wi...


,target_country,shock_id,dimension,dimension_relation,comparisons,mean_target_minus_other,min_target_minus_other,max_target_minus_other
2,DEU,2007,Employment strength,target_weaker,7,-2.4286,-4,-1
5,DEU,2007,Human capital,target_weaker,7,-2.2857,-3,-1
4,DEU,2007,Energy security,target_weaker,5,-1.4000,-2,-1
1,DEU,2007,Debt capacity,target_weaker,5,-1.2000,-2,-1
7,DEU,2007,R&D intensity,target_weaker,4,-1.0000,-1,-1
0,DEU,2007,Debt capacity,equal,2,0.0000,0,0
3,DEU,2007,Energy security,equal,2,0.0000,0,0
6,DEU,2007,R&D intensity,equal,3,0.0000,0,0
11,FRA,2007,Employment strength,target_weaker,5,-2.0000,-3,-1
14,FRA,2007,Human capital,target_weaker,6,-1.5000,-2,-1


In [9]:

# ------------------------------------------------------
# RECOVERY ALIGNMENT FOR TARGET COUNTRIES
# ------------------------------------------------------

target_recovery_alignment = target_base[
    [
        c for c in [
            "Country", "shock_id", "baseline_year", "profile_code", "layer", "is_pareto_frontier",
            "observed_dominance_score_normalized", "observed_incomparability_score_normalized",
            "embedded_separation_score_normalized", "recovery_years", "recovery_status",
        ] if c in target_base.columns
    ]
].copy()

# Add shock median recovery for comparison.
if "recovery_years" in base.columns:
    shock_recovery_median = (
        base.groupby("shock_id")["recovery_years"]
        .median()
        .reset_index(name="shock_median_recovery_years")
    )

    target_recovery_alignment = target_recovery_alignment.merge(
        shock_recovery_median,
        on="shock_id",
        how="left",
    )

    target_recovery_alignment["target_minus_shock_median_recovery"] = (
        pd.to_numeric(target_recovery_alignment["recovery_years"], errors="coerce")
        - pd.to_numeric(target_recovery_alignment["shock_median_recovery_years"], errors="coerce")
    )

    target_recovery_alignment["recovery_alignment_label"] = np.select(
        [
            target_recovery_alignment["target_minus_shock_median_recovery"] < 0,
            target_recovery_alignment["target_minus_shock_median_recovery"] > 0,
            target_recovery_alignment["target_minus_shock_median_recovery"] == 0,
        ],
        [
            "faster_than_shock_median",
            "slower_than_shock_median",
            "equal_to_shock_median",
        ],
        default="unknown",
    )

save_table(
    target_recovery_alignment,
    "target_recovery_alignment.csv",
    "Target recovery alignment",
    "Target country POSet position compared with GDP recovery outcome.",
)

display(target_recovery_alignment)


Saved table: target_recovery_alignment.csv


,Country,shock_id,baseline_year,profile_code,layer,is_pareto_frontier,observed_dominance_score_normalized,observed_incomparability_score_normalized,embedded_separation_score_normalized,recovery_years,recovery_status,shock_median_recovery_years,target_minus_shock_median_recovery,recovery_alignment_label
0,CAN,2007,2007,3-3-4-5-5,1,True,0.3333,0.6667,0.1607,1.0000,recovered,5.0000,-4.0000,faster_than_shock_median
1,DEU,2007,2007,2-1-4-2-3,4,False,0.0417,0.6667,0.0016,2.0000,recovered,5.0000,-3.0000,faster_than_shock_median
2,FRA,2007,2007,2-2-4-3-3,3,False,0.1667,0.5833,0.0128,2.0000,recovered,5.0000,-3.0000,faster_than_shock_median
3,ITA,2007,2007,1-3-2-1-2,3,False,0.0417,0.5417,0.0004,15.0000,recovered,5.0000,10.0000,slower_than_shock_median
4,USA,2007,2007,2-4-5-5-4,1,True,0.3333,0.6667,0.1485,1.0000,recovered,5.0000,-4.0000,faster_than_shock_median
5,CAN,2019,2019,3-3-3-5-5,1,True,0.2424,0.7576,0.1009,1.0000,recovered,1.0000,0.0000,equal_to_shock_median
6,DEU,2019,2019,3-5-5-2-2,1,True,0.1212,0.8788,0.0315,2.0000,recovered,1.0000,1.0000,slower_than_shock_median
7,FRA,2019,2019,1-1-4-3-3,3,False,0.0606,0.7273,0.0030,2.0000,recovered,1.0000,1.0000,slower_than_shock_median
8,ITA,2019,2019,1-1-3-1-1,4,False,0.0000,0.4242,0.0000,2.0000,recovered,1.0000,1.0000,slower_than_shock_median
9,USA,2019,2019,1-5-5-5-5,1,True,0.1818,0.8182,0.0948,1.0000,recovered,1.0000,0.0000,equal_to_shock_median


In [10]:

# ------------------------------------------------------
# COUNTRY-LEVEL REPORT PARAGRAPHS
# ------------------------------------------------------

paragraph_rows = []

for _, row in target_base.iterrows():
    country = row["Country"]
    shock_id = row["shock_id"]
    layer = row.get("layer", np.nan)
    frontier = bool(row.get("is_pareto_frontier", False))
    profile = row.get("profile_code", "")

    dom_count = len(target_dominators_summary[
        target_dominators_summary["target_country"].eq(country)
        & target_dominators_summary["shock_id"].eq(str(shock_id))
    ])

    dominated_count = len(target_dominated_summary[
        target_dominated_summary["target_country"].eq(country)
        & target_dominated_summary["shock_id"].eq(str(shock_id))
    ])

    incomparable_count = len(target_incomparability_summary[
        target_incomparability_summary["target_country"].eq(country)
        & target_incomparability_summary["shock_id"].eq(str(shock_id))
    ])

    frontier_text = "belongs to a Pareto/frontier profile" if frontier else "does not belong to a Pareto/frontier profile"

    recovery_years = row.get("recovery_years", np.nan)

    if pd.notna(recovery_years):
        recovery_text = f"Its GDP recovery outcome is {recovery_years:.1f} years."
    else:
        recovery_text = "Its GDP recovery outcome is not available in the merged validation table."

    paragraph_rows.append({
        "target_country": country,
        "shock_id": shock_id,
        "topic": "Country POSet diagnostic paragraph",
        "report_text": (
            f"For the {shock_id} baseline, {country} is assigned to profile {profile} and layer {layer}. "
            f"It {frontier_text}. In the observed profile POSet, {dom_count} profiles dominate it, "
            f"{dominated_count} profiles are dominated by it, and {incomparable_count} profiles are incomparable with it. "
            f"{recovery_text} This explanation is structural and descriptive, not a causal ranking."
        ),
    })

report_ready_country_drilldown_paragraphs = pd.DataFrame(paragraph_rows)

save_table(
    report_ready_country_drilldown_paragraphs,
    "report_ready_country_drilldown_paragraphs.csv",
    "Report-ready country drilldown paragraphs",
    "Report-ready country-level POSet diagnostic paragraphs for configured targets.",
)

display(report_ready_country_drilldown_paragraphs)


Saved table: report_ready_country_drilldown_paragraphs.csv


,target_country,shock_id,topic,report_text
0,CAN,2007,Country POSet diagnostic paragraph,"For the 2007 baseline, CAN is assigned to prof..."
1,DEU,2007,Country POSet diagnostic paragraph,"For the 2007 baseline, DEU is assigned to prof..."
2,FRA,2007,Country POSet diagnostic paragraph,"For the 2007 baseline, FRA is assigned to prof..."
3,ITA,2007,Country POSet diagnostic paragraph,"For the 2007 baseline, ITA is assigned to prof..."
4,USA,2007,Country POSet diagnostic paragraph,"For the 2007 baseline, USA is assigned to prof..."
5,CAN,2019,Country POSet diagnostic paragraph,"For the 2019 baseline, CAN is assigned to prof..."
6,DEU,2019,Country POSet diagnostic paragraph,"For the 2019 baseline, DEU is assigned to prof..."
7,FRA,2019,Country POSet diagnostic paragraph,"For the 2019 baseline, FRA is assigned to prof..."
8,ITA,2019,Country POSet diagnostic paragraph,"For the 2019 baseline, ITA is assigned to prof..."
9,USA,2019,Country POSet diagnostic paragraph,"For the 2019 baseline, USA is assigned to prof..."


In [11]:

# ------------------------------------------------------
# FIGURES — TARGET COUNTRY DIAGNOSTIC BAR CHARTS
# ------------------------------------------------------

for country in TARGET_COUNTRIES:
    country_df = target_variable_diagnostics[
        target_variable_diagnostics["target_country"].eq(country)
    ].copy()

    if country_df.empty:
        continue

    for shock_id, group in country_df.groupby("shock_id"):
        group = group.sort_values("dimension")

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.bar(group["dimension"], group["ordinal_level_1_to_5"])
        ax.set_ylim(0, 5.5)
        ax.set_ylabel("Ordinal level, 1–5")
        ax.set_title(f"{country} structural profile levels — shock {shock_id}")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(True, axis="y", alpha=0.25)

        save_figure(
            fig,
            f"target_{country}_profile_levels_shock_{shock_id}.png",
            f"{country} profile levels shock {shock_id}",
            "Target country ordinal profile levels across the five final structural variables.",
        )

# Summary figure: target layer by shock
layer_plot = target_base[["Country", "shock_id", "layer"]].dropna().copy()
if len(layer_plot):
    fig, ax = plt.subplots(figsize=(10, 5))

    countries = sorted(layer_plot["Country"].unique())
    x = np.arange(len(countries))
    width = 0.35

    for idx, shock_id in enumerate(sorted(layer_plot["shock_id"].unique())):
        vals = []
        for c in countries:
            sub = layer_plot[layer_plot["Country"].eq(c) & layer_plot["shock_id"].eq(shock_id)]
            vals.append(float(sub["layer"].iloc[0]) if len(sub) else np.nan)

        ax.bar(x + (idx - 0.5) * width, vals, width=width, label=str(shock_id))

    ax.set_xticks(x)
    ax.set_xticklabels(countries)
    ax.set_ylabel("POSet layer")
    ax.set_title("Target country POSet layer by shock")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.25)

    save_figure(
        fig,
        "target_country_layer_by_shock.png",
        "Target country layer by shock",
        "POSet layer comparison for configured target countries.",
    )


Saved figure: target_DEU_profile_levels_shock_2007.png
Saved figure: target_DEU_profile_levels_shock_2019.png
Saved figure: target_ITA_profile_levels_shock_2007.png
Saved figure: target_ITA_profile_levels_shock_2019.png
Saved figure: target_FRA_profile_levels_shock_2007.png
Saved figure: target_FRA_profile_levels_shock_2019.png
Saved figure: target_CAN_profile_levels_shock_2007.png
Saved figure: target_CAN_profile_levels_shock_2019.png
Saved figure: target_USA_profile_levels_shock_2007.png
Saved figure: target_USA_profile_levels_shock_2019.png
Saved figure: target_country_layer_by_shock.png


In [12]:

# ------------------------------------------------------
# INVENTORIES AND AUDIT WORKBOOK
# ------------------------------------------------------

table_inventory_df = pd.DataFrame(table_inventory)
figure_inventory_df = pd.DataFrame(figure_inventory)

for table, file_name in [
    (table_inventory_df, "country_drilldown_table_inventory.csv"),
    (figure_inventory_df, "country_drilldown_figure_inventory.csv"),
]:
    table.to_csv(PROCESSED_DIR / file_name, index=False)
    table.to_csv(DIAGNOSTICS_DIR / file_name, index=False)
    table.to_csv(TABLES_DIR / file_name, index=False)

run_summary = pd.DataFrame([{
    "run_id": RUN_ID,
    "created_at": RUN_TIMESTAMP,
    "target_countries": ",".join(TARGET_COUNTRIES),
    "shock_ids": ",".join(SHOCK_IDS),
    "target_rows": len(target_base),
    "tables_created": len(table_inventory_df),
    "figures_created": len(figure_inventory_df),
    "main_variable_set": MAIN_VARIABLE_SET,
    "main_levels": MAIN_LEVELS,
    "note": "Country drilldown explains POSet position and does not create a scalar index.",
}])

run_summary.to_csv(PROCESSED_DIR / "country_drilldown_run_summary.csv", index=False)
run_summary.to_csv(DIAGNOSTICS_DIR / "country_drilldown_run_summary.csv", index=False)
run_summary.to_csv(TABLES_DIR / "country_drilldown_run_summary.csv", index=False)

audit_xlsx = AUDIT_DIR / "15_Country_POSet_Diagnostic_Drilldown_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    run_summary.to_excel(writer, sheet_name="run_summary", index=False)
    target_base.to_excel(writer, sheet_name="target_brief", index=False)
    target_direct_hasse_parents.to_excel(writer, sheet_name="direct_parents", index=False)
    target_direct_hasse_children.to_excel(writer, sheet_name="direct_children", index=False)
    target_dominators_summary.to_excel(writer, sheet_name="dominators", index=False)
    target_dominated_summary.to_excel(writer, sheet_name="dominated", index=False)
    target_incomparability_summary.to_excel(writer, sheet_name="incomparability", index=False)
    target_variable_diagnostics.to_excel(writer, sheet_name="variable_diagnostics", index=False)
    target_dominators_variable_explanation.to_excel(writer, sheet_name="dominator_variables", index=False)
    target_recovery_alignment.to_excel(writer, sheet_name="recovery_alignment", index=False)
    report_ready_country_drilldown_paragraphs.to_excel(writer, sheet_name="report_paragraphs", index=False)
    table_inventory_df.to_excel(writer, sheet_name="table_inventory", index=False)
    figure_inventory_df.to_excel(writer, sheet_name="figure_inventory", index=False)

print("Audit workbook:", audit_xlsx.resolve())
display(run_summary)
display(table_inventory_df)
display(figure_inventory_df)


Audit workbook: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\15_Country_POSet_Diagnostic_Drilldown_Audit.xlsx


,run_id,created_at,target_countries,shock_ids,target_rows,tables_created,figures_created,main_variable_set,main_levels,note
0,20260624_215532,2026-06-24 21:55:32,"DEU,ITA,FRA,CAN,USA","2007,2019",10,12,11,baseline_5_variables,5,Country drilldown explains POSet position and ...


,file_name,title,description,rows,columns,processed_path,diagnostics_path,table_path,created_at
0,country_poset_diagnostic_drilldown.csv,Country POSet diagnostic drilldown,"Country-level POSet position, profile, diagnos...",60,30,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
1,target_country_problem_brief.csv,Target country problem brief,Compact profile/layer/frontier/recovery summar...,10,30,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
2,target_direct_hasse_parents.csv,Target direct Hasse parents,Direct Hasse parent profiles that dominate tar...,20,7,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
3,target_direct_hasse_children.csv,Target direct Hasse children,Direct Hasse child profiles dominated by targe...,21,7,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
4,target_dominators_summary.csv,Target dominators summary,All profiles that dominate each target country...,49,6,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
5,target_dominated_summary.csv,Target dominated summary,All profiles dominated by each target country ...,42,6,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
6,target_incomparability_summary.csv,Target incomparability summary,All profiles incomparable with each target cou...,194,6,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
7,target_pairwise_variable_gaps.csv,Target pairwise variable gaps,Dimension-level ordinal gaps between target pr...,1425,11,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
8,target_variable_diagnostics.csv,Target variable diagnostics,Final five score/ordinal-level diagnostics for...,50,9,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
9,target_dominators_variable_explanation.csv,Target dominators variable explanation,Dimension-level explanation of where target co...,46,8,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32


,file_name,title,description,path,created_at
0,target_DEU_profile_levels_shock_2007.png,DEU profile levels shock 2007,Target country ordinal profile levels across t...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
1,target_DEU_profile_levels_shock_2019.png,DEU profile levels shock 2019,Target country ordinal profile levels across t...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
2,target_ITA_profile_levels_shock_2007.png,ITA profile levels shock 2007,Target country ordinal profile levels across t...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
3,target_ITA_profile_levels_shock_2019.png,ITA profile levels shock 2019,Target country ordinal profile levels across t...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
4,target_FRA_profile_levels_shock_2007.png,FRA profile levels shock 2007,Target country ordinal profile levels across t...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
5,target_FRA_profile_levels_shock_2019.png,FRA profile levels shock 2019,Target country ordinal profile levels across t...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
6,target_CAN_profile_levels_shock_2007.png,CAN profile levels shock 2007,Target country ordinal profile levels across t...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
7,target_CAN_profile_levels_shock_2019.png,CAN profile levels shock 2019,Target country ordinal profile levels across t...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
8,target_USA_profile_levels_shock_2007.png,USA profile levels shock 2007,Target country ordinal profile levels across t...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32
9,target_USA_profile_levels_shock_2019.png,USA profile levels shock 2019,Target country ordinal profile levels across t...,D:\Milano Bicocca\Course Materials\1st Year\02...,2026-06-24 21:55:32


In [13]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

expected_outputs = [
    "country_poset_diagnostic_drilldown.csv",
    "target_country_problem_brief.csv",
    "target_direct_hasse_parents.csv",
    "target_direct_hasse_children.csv",
    "target_dominators_summary.csv",
    "target_dominated_summary.csv",
    "target_incomparability_summary.csv",
    "target_pairwise_variable_gaps.csv",
    "target_variable_diagnostics.csv",
    "target_dominators_variable_explanation.csv",
    "target_recovery_alignment.csv",
    "report_ready_country_drilldown_paragraphs.csv",
    "country_drilldown_run_summary.csv",
]

output_check = pd.DataFrame([
    {
        "file_name": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_outputs
])

output_check.to_csv(DIAGNOSTICS_DIR / "output_check.csv", index=False)

print("15 COUNTRY POSET DIAGNOSTIC DRILLDOWN COMPLETE")
print("=" * 90)

display(output_check)

print("\nImportant interpretation:")
print("This drilldown explains country-specific POSet position. It is not a scalar index/ranking.")

print("\nKey outputs to inspect/send:")
print("- 15_Country_POSet_Diagnostic_Drilldown_Audit.xlsx")
print("- target_country_problem_brief.csv")
print("- target_direct_hasse_parents.csv")
print("- target_direct_hasse_children.csv")
print("- target_variable_diagnostics.csv")
print("- target_recovery_alignment.csv")

print("\nNext notebook:")
print("16_Multi_Indicator_Shock_Validation_2002_5Var.ipynb")


15 COUNTRY POSET DIAGNOSTIC DRILLDOWN COMPLETE


,file_name,processed_exists,diagnostics_exists,tables_exists
0,country_poset_diagnostic_drilldown.csv,True,True,True
1,target_country_problem_brief.csv,True,True,True
2,target_direct_hasse_parents.csv,True,True,True
3,target_direct_hasse_children.csv,True,True,True
4,target_dominators_summary.csv,True,True,True
5,target_dominated_summary.csv,True,True,True
6,target_incomparability_summary.csv,True,True,True
7,target_pairwise_variable_gaps.csv,True,True,True
8,target_variable_diagnostics.csv,True,True,True
9,target_dominators_variable_explanation.csv,True,True,True



Important interpretation:
This drilldown explains country-specific POSet position. It is not a scalar index/ranking.

Key outputs to inspect/send:
- 15_Country_POSet_Diagnostic_Drilldown_Audit.xlsx
- target_country_problem_brief.csv
- target_direct_hasse_parents.csv
- target_direct_hasse_children.csv
- target_variable_diagnostics.csv
- target_recovery_alignment.csv

Next notebook:
16_Multi_Indicator_Shock_Validation_2002_5Var.ipynb
